In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Assignment 3: Learning to Play Snake with Q-Learning

The instructions are in the assignment PDF. Write your code where it says `# YOUR CODE HERE`, answer the questions at the end, and do not change the setup cell or the settings.

Before you submit, choose **Runtime > Restart session and run all** and check that every cell runs without errors.

In [4]:
#@title Setup: run this cell first
from pathlib import Path

SOURCES = {}
SOURCES['snake_env.py'] = r'''"""Snake on a small board, for tabular Q-Learning.

The board has SIZE x SIZE cells and everything outside it is wall. Positions are
(x, y): x grows to the right and y grows downwards, so "up" means a smaller y.

    env = SnakeEnv(seed=0)
    state = env.reset()
    next_state, reward, done, info = env.step(STRAIGHT)

A state is a tuple of 11 integers, each 0 or 1, in the order of STATE_FEATURES.
"""
import random

SIZE = 8
START_LENGTH = 3
MAX_STEPS_WITHOUT_FOOD = 40

REWARD_FOOD = 10.0
REWARD_DEATH = -10.0
REWARD_TIMEOUT = -5.0
REWARD_STEP = -0.1

STRAIGHT, TURN_LEFT, TURN_RIGHT = 0, 1, 2
ACTIONS = (STRAIGHT, TURN_LEFT, TURN_RIGHT)
ACTION_NAMES = ("straight", "turn_left", "turn_right")

UP, RIGHT, DOWN, LEFT = (0, -1), (1, 0), (0, 1), (-1, 0)
CLOCKWISE = (UP, RIGHT, DOWN, LEFT)
DIRECTION_NAMES = {UP: "up", RIGHT: "right", DOWN: "down", LEFT: "left"}

STATE_FEATURES = (
    "danger_straight", "danger_left", "danger_right",
    "moving_up", "moving_down", "moving_left", "moving_right",
    "food_up", "food_down", "food_left", "food_right",
)


class SnakeEnv:
    """One game of Snake. Call reset() to start an episode and step(action) to play it.

    Attributes you may read (do not change them):
        snake       list of (x, y) cells, head first
        food        (x, y) of the food, or None if the board is full
        direction   the direction of the last move, one of UP, RIGHT, DOWN, LEFT
        food_eaten  food eaten in this episode
        steps       steps played in this episode
        end_reason  None while playing, then "wall", "body", "timeout" or "full"
    """

    def __init__(self, size=SIZE, max_steps_without_food=MAX_STEPS_WITHOUT_FOOD, seed=None):
        if size < START_LENGTH + 2:
            raise ValueError(f"size must be at least {START_LENGTH + 2}")
        self.size = size
        self.max_steps_without_food = max_steps_without_food
        self.rng = random.Random(seed)
        self.reset()

    def reset(self):
        """Start a new episode and return its first state."""
        middle = self.size // 2
        self.direction = RIGHT
        self.snake = [(middle - i, middle) for i in range(START_LENGTH)]
        self.food = self._random_free_cell()
        self.food_eaten = 0
        self.steps = 0
        self.steps_without_food = 0
        self.total_reward = 0.0
        self.end_reason = None
        return self.get_state()

    def step(self, action):
        """Play one move and return (next_state, reward, done, info).

        info is a dict with "ate" (True if the snake ate on this step) and
        "reason" (None, or why the episode ended).
        """
        if self.end_reason is not None:
            raise RuntimeError("The episode is over: call reset() first.")
        if action not in ACTIONS:
            raise ValueError(f"action must be 0, 1 or 2, got {action!r}")
        self.direction = self._turned(int(action))
        head = self._ahead(self.direction)
        self.steps += 1
        ate = False

        if self._blocked(head):
            self.end_reason = "wall" if self._outside(head) else "body"
            reward = REWARD_DEATH
        else:
            self.snake.insert(0, head)
            if head == self.food:
                ate = True
                self.food_eaten += 1
                self.steps_without_food = 0
                reward = REWARD_FOOD
                self.food = self._random_free_cell()
                if self.food is None:
                    self.end_reason = "full"
            else:
                self.snake.pop()
                self.steps_without_food += 1
                reward = REWARD_STEP
                if self.steps_without_food >= self.max_steps_without_food:
                    self.end_reason = "timeout"
                    reward = REWARD_TIMEOUT

        self.total_reward += reward
        done = self.end_reason is not None
        return self.get_state(), reward, done, {"ate": ate, "reason": self.end_reason}

    def get_state(self):
        """The 11 binary features, in the order of STATE_FEATURES."""
        index = CLOCKWISE.index(self.direction)
        left, right = CLOCKWISE[index - 1], CLOCKWISE[(index + 1) % 4]
        head_x, head_y = self.snake[0]
        food_x, food_y = self.food if self.food is not None else (head_x, head_y)
        features = (
            self._blocked(self._ahead(self.direction)),
            self._blocked(self._ahead(left)),
            self._blocked(self._ahead(right)),
            self.direction == UP,
            self.direction == DOWN,
            self.direction == LEFT,
            self.direction == RIGHT,
            food_y < head_y,
            food_y > head_y,
            food_x < head_x,
            food_x > head_x,
        )
        return tuple(int(value) for value in features)

    def render(self):
        """The board as text: O is the head, o the body, * the food and # the wall."""
        rows = ["#" * (self.size + 2)]
        for y in range(self.size):
            row = ""
            for x in range(self.size):
                cell = (x, y)
                if cell == self.snake[0]:
                    row += "O"
                elif cell in self.snake:
                    row += "o"
                elif cell == self.food:
                    row += "*"
                else:
                    row += "."
            rows.append("#" + row + "#")
        rows.append("#" * (self.size + 2))
        return "\n".join(rows)

    def _turned(self, action):
        index = CLOCKWISE.index(self.direction)
        if action == TURN_LEFT:
            return CLOCKWISE[index - 1]
        if action == TURN_RIGHT:
            return CLOCKWISE[(index + 1) % 4]
        return self.direction

    def _ahead(self, direction):
        head_x, head_y = self.snake[0]
        return head_x + direction[0], head_y + direction[1]

    def _outside(self, cell):
        return not (0 <= cell[0] < self.size and 0 <= cell[1] < self.size)

    def _blocked(self, cell):
        # The tail moves away on the same step, so moving into it is safe.
        return self._outside(cell) or cell in self.snake[:-1]

    def _random_free_cell(self):
        occupied = set(self.snake)
        free = [(x, y) for y in range(self.size) for x in range(self.size) if (x, y) not in occupied]
        return self.rng.choice(free) if free else None
'''
SOURCES['snake_utils.py'] = r'''"""Helpers for the notebook: basic checks, plots, a replay of one episode and results.json."""
import functools
import json
import random

from snake_env import ACTIONS, SnakeEnv

RESULT_KEYS = ("avg_food", "avg_reward", "avg_steps", "max_food", "deaths", "timeouts")
HISTORY_KEYS = ("food", "reward", "steps", "epsilon")


def _keeps_random_state(check):
    """The checks use their own seeds and then put back the notebook's random state."""

    @functools.wraps(check)
    def wrapper(*args, **kwargs):
        saved = random.getstate()
        try:
            return check(*args, **kwargs)
        finally:
            random.setstate(saved)

    return wrapper


def _passed(task):
    print(f"{task}: basic checks passed. They do not cover everything, so test your own cases too.")


@_keeps_random_state
def check_choose_action(choose_action):
    state = (0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1)
    Q = {state: [0.0, 5.0, 1.0]}
    for _ in range(20):
        action = choose_action(Q, state, 0.0)
        assert action == 1, f"with epsilon = 0 the best action is 1, but choose_action returned {action!r}"

    unseen = (1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0)
    action = choose_action(Q, unseen, 0.0)
    assert action in ACTIONS, f"choose_action must return 0, 1 or 2, got {action!r}"
    assert unseen in Q, "an unseen state must be added to Q"
    assert len(Q[unseen]) == 3 and all(value == 0 for value in Q[unseen]), "an unseen state must start with 0.0 for each action"

    chosen = {choose_action(Q, state, 1.0) for _ in range(300)}
    assert chosen == set(ACTIONS), f"with epsilon = 1 every action should appear, but only {sorted(chosen)} did"
    _passed("Task 1")


@_keeps_random_state
def check_update_q_value(update_q_value):
    state, next_state = (0,) * 11, (1,) + (0,) * 10
    Q = {state: [0.0, 0.0, 0.0], next_state: [1.0, 2.0, 0.5]}
    new = update_q_value(Q, state, 0, -0.1, next_state, False, 0.5, 0.9)
    expected = 0.5 * (-0.1 + 0.9 * 2.0)
    assert abs(Q[state][0] - expected) < 1e-9, f"Q[state][0] should be {expected:.4f}, got {Q[state][0]!r}"
    assert new is not None and abs(new - expected) < 1e-9, f"update_q_value should return the new value {expected:.4f}, got {new!r}"
    assert list(Q[next_state]) == [1.0, 2.0, 0.5], "the next state's values must not change"
    _passed("Task 2")


@_keeps_random_state
def check_train_agent(train_agent):
    random.seed(0)
    output = train_agent(SnakeEnv(seed=0), 50, 0.1, 0.9, 1.0, 0.05)
    assert isinstance(output, tuple) and len(output) == 2, "train_agent must return (Q, history)"
    Q, history = output
    assert isinstance(Q, dict) and Q, "Q must be a non-empty dict after training"
    assert isinstance(history, dict), "history must be a dict"
    for key in HISTORY_KEYS:
        assert key in history, f'history is missing "{key}"'
        assert len(history[key]) == 50, f'history["{key}"] must have one value per episode (50), got {len(history[key])}'
    epsilon = history["epsilon"]
    assert abs(epsilon[0] - 1.0) < 1e-9 and abs(epsilon[-1] - 0.05) < 1e-9, "epsilon must go from epsilon_start to epsilon_end"
    _passed("Task 3")


@_keeps_random_state
def check_evaluate_agent(evaluate_agent):
    random.seed(0)
    results = evaluate_agent(SnakeEnv(seed=0), {}, 20, 1.0)
    assert isinstance(results, dict), "evaluate_agent must return a dict"
    for key in RESULT_KEYS:
        assert key in results, f'the results are missing "{key}"'
    assert results["deaths"] + results["timeouts"] == 20, "with a random agent, every episode should end in a death or a timeout"
    assert results["avg_food"] <= results["max_food"], "avg_food cannot be larger than max_food"
    _passed("Task 4")


def plot_training(history, window=100):
    """Food per episode (moving average) and epsilon, over training."""
    import matplotlib.pyplot as plt

    food = history["food"]
    window = max(1, min(window, len(food)))
    averages = [sum(food[max(0, i - window + 1):i + 1]) / (i + 1 - max(0, i - window + 1)) for i in range(len(food))]
    figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))
    left.plot(averages)
    left.set_xlabel("Episode")
    left.set_ylabel(f"Food eaten (average of last {window})")
    right.plot(history["epsilon"], color="tab:orange")
    right.set_xlabel("Episode")
    right.set_ylabel("Epsilon")
    figure.tight_layout()
    plt.show()


def show_comparison(trained, random_agent):
    """Print the trained agent and the random agent side by side."""
    print(f"{'':12}{'TRAINED':>10}{'RANDOM':>10}")
    for key in RESULT_KEYS:
        if key.startswith("avg_"):
            print(f"{key:12}{trained[key]:>10.2f}{random_agent[key]:>10.2f}")
        else:
            print(f"{key:12}{int(trained[key]):>10}{int(random_agent[key]):>10}")


def watch_episode(Q, choose_action, seed=None, max_steps=300):
    """Replay one greedy episode (epsilon = 0) as an animation."""
    import matplotlib.pyplot as plt
    from matplotlib import animation
    from IPython.display import HTML, display

    env = SnakeEnv(seed=seed)
    state = env.reset()
    frames = [(list(env.snake), env.food, env.food_eaten)]
    done = False
    while not done and len(frames) <= max_steps:
        state, _, done, _ = env.step(choose_action(Q, state, 0.0))
        frames.append((list(env.snake), env.food, env.food_eaten))

    size = env.size
    figure, axis = plt.subplots(figsize=(3.6, 3.9), dpi=60)

    def draw(index):
        snake, food, eaten = frames[index]
        grid = [[0] * size for _ in range(size)]
        for x, y in snake[1:]:
            grid[y][x] = 1
        x, y = snake[0]
        grid[y][x] = 2
        if food is not None:
            grid[food[1]][food[0]] = 3
        axis.clear()
        axis.imshow(grid, cmap=plt.matplotlib.colors.ListedColormap(["#f4f4f4", "#5aa05a", "#1f5f1f", "#d64545"]), vmin=0, vmax=3)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_xticks([i - 0.5 for i in range(1, size)], minor=True)
        axis.set_yticks([i - 0.5 for i in range(1, size)], minor=True)
        axis.grid(which="minor", color="white", linewidth=1)
        axis.tick_params(which="minor", length=0)
        axis.set_title(f"Step {index}   Food {eaten}")

    movie = animation.FuncAnimation(figure, draw, frames=len(frames), interval=120)
    plt.close(figure)
    display(HTML(movie.to_jshtml()))
    print(f"The episode ended after {env.steps} steps with {env.food_eaten} food ({env.end_reason}).")


def _rounded(results):
    return {key: round(float(results[key]), 4) if key.startswith("avg_") else int(results[key]) for key in RESULT_KEYS}


def save_results(path, episodes, alpha, gamma, epsilon_start, epsilon_end, trained, random_agent):
    """Write results.json in the format I grade."""
    for name, results in (("trained", trained), ("random_agent", random_agent)):
        missing = [key for key in RESULT_KEYS if key not in results]
        if missing:
            raise ValueError(f"{name} is missing {missing}")
    data = {
        "episodes": episodes,
        "alpha": alpha,
        "gamma": gamma,
        "epsilon_start": epsilon_start,
        "epsilon_end": epsilon_end,
        "evaluation": _rounded(trained),
        "random_baseline": _rounded(random_agent),
    }
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)
    print(f"Saved {path}")
'''
for name, source in SOURCES.items():
    Path(name).write_text(source, encoding="utf-8")

import random
import sys
import numpy as np

sys.path.insert(0, ".")
from snake_env import SnakeEnv, ACTIONS, ACTION_NAMES, STATE_FEATURES, STRAIGHT, TURN_LEFT, TURN_RIGHT
from snake_utils import check_choose_action, check_update_q_value, check_train_agent, check_evaluate_agent
from snake_utils import plot_training, show_comparison, watch_episode, save_results
print("Environment ready.")

Environment ready.


## The environment

`env.reset()` starts an episode and returns the first state. `env.step(action)` plays one move and returns `(next_state, reward, done, info)`. A state is a tuple of 11 values, each 0 or 1. The actions are `0` (straight), `1` (turn left) and `2` (turn right), relative to the direction the snake is moving.

Run the cell below to see one move.

In [5]:
env = SnakeEnv(seed=0)
state = env.reset()
print(env.render())
print(dict(zip(STATE_FEATURES, state)))

next_state, reward, done, info = env.step(STRAIGHT)
print("reward:", reward, " done:", done, " info:", info)
print(env.render())

##########
#........#
#........#
#........#
#*.......#
#..ooO...#
#........#
#........#
#........#
##########
{'danger_straight': 0, 'danger_left': 0, 'danger_right': 0, 'moving_up': 0, 'moving_down': 0, 'moving_left': 0, 'moving_right': 1, 'food_up': 1, 'food_down': 0, 'food_left': 1, 'food_right': 0}
reward: -0.1  done: False  info: {'ate': False, 'reason': None}
##########
#........#
#........#
#........#
#*.......#
#...ooO..#
#........#
#........#
#........#
##########


## Settings
Keep these values in the notebook you submit.

In [6]:
SEED = 42
EPISODES = 5000
ALPHA = 0.1
GAMMA = 0.9
EPSILON_START = 1.0
EPSILON_END = 0.05
EVAL_EPISODES = 200

random.seed(SEED)
np.random.seed(SEED)

## Task 1: Epsilon-greedy action selection

The Q-table `Q` is a dict that maps each state to a list of three values, one per action, so `Q[state][action]` is the value of an action in a state.

Complete `choose_action`:

- If `state` is not in `Q` yet, add it with `0.0` for each action.
- With probability `epsilon`, return one of the three actions at random.
- Otherwise, return the action with the highest value in `Q[state]`. If several actions share the highest value, any of them is fine.

Use Python's `random` module for the random choices. The settings cell seeds it, so running the notebook again gives the same results.

In [ ]:
def choose_action(Q, state, epsilon):
    """
    Return an action (0, 1 or 2) chosen with an epsilon-greedy strategy.

    Q        dict that maps a state to a list of 3 values, one per action
    state    the current state, a tuple of 11 integers
    epsilon  probability of choosing a random action, from 0 to 1
    """
   
    list_of_actions = [0, 1, 2]
    if state not in Q:
        Q[state] = [0.0, 0.0, 0.0]
    if random.random() < epsilon:
        return random.choice(list_of_actions)
    else:
        return int(np.argmax(Q[state]))

In [10]:
check_choose_action(choose_action)

Task 1: basic checks passed. They do not cover everything, so test your own cases too.


## Task 2: Q-Learning update

Complete `update_q_value` with the Q-Learning rule:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

- If `done` is `True`, the episode has ended and there is no future to add: use $r$ in place of $r + \gamma \max_{a'} Q(s',a')$.
- If `state` or `next_state` is not in `Q` yet, add it with `0.0` for each action.
- Change `Q` in place and return the new value of `Q[state][action]`.

In [ ]:
def update_q_value(Q, state, action, reward, next_state, done, alpha, gamma):
    """Apply one Q-Learning update to Q[state][action] and return its new value."""
    
    reward = float(reward)
    if state not in Q:
        Q[state] = [0.0, 0.0, 0.0]
    if next_state not in Q:
        Q[next_state] = [0.0, 0.0, 0.0]
    if done:
        Q[state][action] += alpha * (reward - Q[state][action])
    else:
        Q[state][action] += alpha * (reward + gamma * max(Q[next_state]) - Q[state][action])
    return Q[state][action]

In [19]:
check_update_q_value(update_q_value)

Task 2: basic checks passed. They do not cover everything, so test your own cases too.


## Task 3: Training

Complete `train_agent`:

- Start from an empty Q-table, `Q = {}`.
- Play `episodes` episodes. Each one starts with `env.reset()` and ends when `env.step()` returns `done = True`.
- At every step, choose the action with `choose_action` and learn from the result with `update_q_value`.
- Decrease epsilon linearly, from `epsilon_start` in the first episode to `epsilon_end` in the last one. In episode `i`, counting from 0, of `N` episodes: `epsilon = epsilon_start + (epsilon_end - epsilon_start) * i / (N - 1)`. If `N` is 1, use `epsilon_start`.
- Return `(Q, history)`, where `history` is a dict of lists with one value per episode: `"food"` (food eaten, `env.food_eaten` at the end of the episode), `"reward"` (total reward), `"steps"` (steps played) and `"epsilon"` (the epsilon used).

In [ ]:
def train_agent(env, episodes, alpha, gamma, epsilon_start, epsilon_end):
    """Train a Q-table with Q-Learning and return (Q, history)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check_train_agent(train_agent)

Now train your agent with the settings above. It should take well under a minute.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
env = SnakeEnv(seed=SEED)
Q, history = train_agent(env, EPISODES, ALPHA, GAMMA, EPSILON_START, EPSILON_END)
print("States in the Q-table:", len(Q))
plot_training(history)

## Task 4: Evaluation

Complete `evaluate_agent`. It plays `episodes` episodes choosing actions with `choose_action` and the given `epsilon`. It does not learn: it never calls `update_q_value`, and the values already in `Q` must not change. (`choose_action` may still add unseen states with zeros, and that is fine.) It returns a dict with:

- `"avg_food"`: average food eaten per episode
- `"avg_reward"`: average total reward per episode
- `"avg_steps"`: average number of steps per episode
- `"max_food"`: the most food eaten in one episode
- `"deaths"`: episodes that ended against a wall or the snake's body
- `"timeouts"`: episodes that ended for too many steps without food

A random agent is the same function with an empty Q-table and `epsilon = 1`.

In [ ]:
def evaluate_agent(env, Q, episodes=200, epsilon=0.0):
    """Play episodes without learning and return a dict with the results."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check_evaluate_agent(evaluate_agent)

The next cell compares your trained agent with a random agent and writes `results.json`. Download it from the Files panel and submit it with this notebook.

In [ ]:
random.seed(SEED)
trained = evaluate_agent(SnakeEnv(seed=SEED + 1), Q, EVAL_EPISODES, epsilon=0.0)
random.seed(SEED)
random_agent = evaluate_agent(SnakeEnv(seed=SEED + 1), {}, EVAL_EPISODES, epsilon=1.0)
show_comparison(trained, random_agent)
save_results("results.json", EPISODES, ALPHA, GAMMA, EPSILON_START, EPSILON_END, trained, random_agent)

Watch your agent play one episode. Drawing the replay takes a few seconds.

In [ ]:
watch_episode(Q, choose_action, seed=SEED + 2)

Results of your trained agent with different values of epsilon during evaluation:

In [ ]:
for epsilon in (0.0, 0.1, 0.3, 1.0):
    random.seed(SEED)
    results = evaluate_agent(SnakeEnv(seed=SEED + 1), Q, EVAL_EPISODES, epsilon=epsilon)
    print(f"epsilon {epsilon}: avg_food {results['avg_food']:.2f}, deaths {results['deaths']}")

## Task 5: Short answers
Answer each question in two or three sentences, in the cell below it.

**Question 1.** Why is exploration necessary at the beginning of training?

*Your answer here.*

**Question 2.** What happens to your trained agent's results when you evaluate it with a high epsilon, and why? Use the numbers from the comparison of epsilon values above.

*Your answer here.*

**Question 3.** Give one limitation of the state representation used in this assignment, and one situation in the game where it causes the snake to make a bad move.

*Your answer here.*